In [1]:
from pathlib import Path

repo_path = Path("/content/AudioSep")
if not repo_path.exists():
    !git clone https://github.com/Audio-AGI/AudioSep.git

%cd /content/AudioSep

/content/AudioSep


In [2]:
!pip install torchlibrosa==0.1.0 gradio==3.47.1 gdown lightning transformers ftfy braceexpand webdataset soundfile wget h5py

In [3]:
checkpoints_dir = Path("checkpoint")
checkpoints_dir.mkdir(exist_ok=True)

models = (
    (
        "https://huggingface.co/spaces/badayvedat/AudioSep/resolve/main/checkpoint/audiosep_base_4M_steps.ckpt",
        checkpoints_dir / "audiosep_base_4M_steps.ckpt"
    ),
    (
        "https://huggingface.co/spaces/badayvedat/AudioSep/resolve/main/checkpoint/music_speech_audioset_epoch_15_esc_89.98.pt",
        checkpoints_dir / "music_speech_audioset_epoch_15_esc_89.98.pt"
    )
)

for model_url, model_path in models:
    if not model_path.exists():
        !wget {model_url} -O {model_path}

In [4]:
!wget "https://audio-agi.github.io/Separate-Anything-You-Describe/demos/exp31_water drops_mixture.wav"

--2026-04-21 21:24:47--  https://audio-agi.github.io/Separate-Anything-You-Describe/demos/exp31_water%20drops_mixture.wav
Resolving audio-agi.github.io (audio-agi.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to audio-agi.github.io (audio-agi.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 320044 (313K) [audio/wav]
Saving to: ‘exp31_water drops_mixture.wav.1’

exp31_water drops_m 100%[===================>] 312.54K  --.-KB/s    in 0.03s   

2026-04-21 21:24:48 (9.45 MB/s) - ‘exp31_water drops_mixture.wav.1’ saved [320044/320044]



In [5]:
import torch
from pipeline import build_audiosep, separate_audio

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = build_audiosep(
      config_yaml='config/audiosep_base.yaml',
      checkpoint_path=str(models[0][1]),
      device=device)

audio_file = 'exp31_water drops_mixture.wav'
text = 'water drops'
output_file='separated_audio.wav'

# AudioSep processes the audio at 32 kHz sampling rate
separate_audio(model, audio_file, text, output_file, device)

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RuntimeError: Error(s) in loading state_dict for CLAP:
	Unexpected key(s) in state_dict: "text_branch.embeddings.position_ids". 

In [ ]:
print(f"The separated audio is saved to: '{output_file}' file.")